# Phase 1: Delay EDA

Starting point for exploring `v_stop_delays` / `v_route_otp_daily`. Fill in
as you go -- this is meant to be a launch pad, not a finished analysis.

Questions to look at first:
1. What does the overall arrival-delay distribution look like (mean/median/tails)?
2. Does delay vary by hour of day / day of week?
3. Which routes/operators have the worst on-time performance?
4. Does delay grow along a trip (later stops worse than earlier stops)? This
   matters a lot for Phase 4 -- if so, upstream delay-so-far is a strong,
   *leakage-free* predictive feature (known at prediction time), unlike
   downstream stop info.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
engine = create_engine(os.environ["DATABASE_URL"])

In [ ]:
delays = pd.read_sql("select * from v_stop_delays", engine)
delays["arrival_delay_min"] = delays["arrival_delay_sec"] / 60
delays["arrival_delay_min"].describe()

In [ ]:
delays["arrival_delay_min"].clip(-10, 30).hist(bins=80)

In [ ]:
otp = pd.read_sql("select * from v_route_otp_daily", engine)
otp.groupby("route_short_name")["on_time_rate"].mean().sort_values().head(20)

In [ ]:
# Delay progression along a trip: join stop_sequence and look at whether
# later stops in the same trip tend to be later-delayed than earlier ones.
prog = delays.groupby("stop_sequence")["arrival_delay_sec"].mean()
prog.head(30)